# Lastnames CCS230 Finals: Heart Disease Data Mining Project

This notebook is a beginner-friendly data mining final project using `heart.csv`. The goal is to explore patient information, discover useful patterns, group patients into archetypes, and build classification models that predict the `HeartDisease` outcome.

**Project workflow:**
1. Import libraries
2. Load and inspect dataset
3. Exploratory Data Analysis
4. Data preparation and encoding
5. Association Rule Mining
6. K-Means clustering for patient archetypes
7. Decision Tree classification
8. Random Forest classification
9. Model evaluation and comparison
10. Final clinical insights and recommendation

**Important note:** This project is for educational data mining practice only. The results should not be used as medical advice or as a replacement for clinical diagnosis.

## 1. Import Libraries

We first import the Python libraries needed for data analysis, visualization, association rule mining, clustering, classification, and model evaluation.

In [ ]:
# Basic data handling
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data preparation and machine learning
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Association rule mining. Install mlxtend if this import fails.
try:
    from mlxtend.frequent_patterns import apriori, association_rules
except ImportError:
    apriori = None
    association_rules = None

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

# Fixed random state makes the results reproducible.
RANDOM_STATE = 42

## 2. Load and Inspect Dataset

This section loads `heart.csv` and checks the structure of the dataset. Before modeling, we need to understand the number of rows, column names, data types, missing values, duplicate rows, and target distribution.

In [ ]:
# Load the dataset from the same folder as this notebook.
data_path = Path("heart.csv")

if not data_path.exists():
    raise FileNotFoundError("heart.csv was not found. Please place it in the same folder as this notebook.")

df = pd.read_csv(data_path)

# Show the first few rows to confirm that the file loaded correctly.
df.head()

In [ ]:
# Check the size of the dataset and the names of all columns.
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# View data types and non-null counts.
# This helps us identify numerical and categorical variables.
df.info()

In [ ]:
# Check missing values and duplicate rows.
inspection_summary = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
})

display(inspection_summary)
print(f"Duplicate rows: {df.duplicated().sum()}")

### Dataset Dictionary

The dataset uses a mix of numerical and categorical patient attributes:

- `Age`: patient age
- `Sex`: biological sex recorded as `M` or `F`
- `ChestPainType`: type of chest pain
- `RestingBP`: resting blood pressure
- `Cholesterol`: serum cholesterol
- `FastingBS`: fasting blood sugar indicator
- `RestingECG`: resting electrocardiogram result
- `MaxHR`: maximum heart rate achieved
- `ExerciseAngina`: exercise-induced angina indicator
- `Oldpeak`: ST depression induced by exercise
- `ST_Slope`: slope of the peak exercise ST segment
- `HeartDisease`: target variable, where `1` means heart disease is present and `0` means heart disease is not present

## 3. Exploratory Data Analysis

Exploratory Data Analysis, or EDA, helps us understand patterns before applying data mining models. We will look at summary statistics, target class balance, categorical variable distributions, numerical variable distributions, and correlations.

In [ ]:
# Summary statistics for both numerical and categorical columns.
df.describe(include="all").T

In [ ]:
# Target distribution shows whether the classes are balanced or imbalanced.
target_counts = df["HeartDisease"].value_counts().sort_index()
target_percent = df["HeartDisease"].value_counts(normalize=True).sort_index() * 100

target_summary = pd.DataFrame({
    "count": target_counts,
    "percent": target_percent.round(2)
})

display(target_summary)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="HeartDisease")
plt.title("Heart Disease Target Distribution")
plt.xlabel("HeartDisease (0 = No, 1 = Yes)")
plt.ylabel("Number of Patients")
plt.show()

In [ ]:
# Separate numerical and categorical columns for easier analysis.
target_col = "HeartDisease"
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop(target_col).tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

In [ ]:
# Plot categorical variables against the target.
# These charts help reveal which categories are associated with more heart disease cases.
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, col in zip(axes, categorical_cols):
    sns.countplot(data=df, x=col, hue="HeartDisease", ax=ax)
    ax.set_title(f"{col} by HeartDisease")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=20)

# Hide unused subplot if there are fewer categorical columns than axes.
for ax in axes[len(categorical_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Plot numerical variables to compare their distributions by target class.
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, col in zip(axes, numeric_cols):
    sns.histplot(data=df, x=col, hue="HeartDisease", kde=True, bins=25, ax=ax)
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)

for ax in axes[len(numeric_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for numerical columns.
# Correlation values close to 1 or -1 indicate stronger linear relationships.
plt.figure(figsize=(10, 7))
corr = df[numeric_cols + [target_col]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

## 4. Data Preparation and Encoding

Machine learning algorithms usually require numerical input. We will split the dataset into features and target, then use one-hot encoding for categorical columns and scaling for numerical columns.

For classification, we use a train-test split so the models are evaluated on data they did not learn from.

In [ ]:
# Separate input features X from the target y.
X = df.drop(columns=[target_col])
y = df[target_col]

# Stratify keeps the same target proportion in training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows: {X_test.shape[0]}")

In [ ]:
# Preprocessing for classification models.
# Numerical values are scaled; categorical values are converted into one-hot columns.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

# A helper function to get the transformed feature names after preprocessing.
def get_feature_names(preprocessor):
    num_features = preprocessor.named_transformers_["num"].get_feature_names_out(numeric_cols)
    cat_features = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols)
    return np.concatenate([num_features, cat_features])

## 5. Association Rule Mining

Association Rule Mining finds patterns in the form: **if these items appear together, another item is likely to appear too**. In this project, we convert patient attributes into item-like labels, such as `Age=Senior`, `ChestPainType=ASY`, or `HeartDisease=Yes`.

We will focus on rules that have `HeartDisease=Yes` as the consequence, because those rules may describe combinations of patient characteristics associated with heart disease.

In [ ]:
# Create a copy for association rule mining.
arm_df = df.copy()

# Convert numerical variables into readable bins.
# Binning makes continuous values easier to treat as transaction items.
arm_df["AgeGroup"] = pd.cut(
    arm_df["Age"],
    bins=[0, 40, 55, 70, 120],
    labels=["Age=Young", "Age=Middle", "Age=Older", "Age=Senior"]
)

arm_df["RestingBPGroup"] = pd.cut(
    arm_df["RestingBP"],
    bins=[0, 120, 140, 300],
    labels=["RestingBP=Normal", "RestingBP=Elevated", "RestingBP=High"]
)

arm_df["CholesterolGroup"] = pd.cut(
    arm_df["Cholesterol"],
    bins=[-1, 0, 200, 240, 700],
    labels=["Cholesterol=UnknownOrZero", "Cholesterol=Desirable", "Cholesterol=Borderline", "Cholesterol=High"]
)

arm_df["MaxHRGroup"] = pd.cut(
    arm_df["MaxHR"],
    bins=[0, 120, 150, 250],
    labels=["MaxHR=Low", "MaxHR=Moderate", "MaxHR=High"]
)

arm_df["OldpeakGroup"] = pd.cut(
    arm_df["Oldpeak"],
    bins=[-5, 0, 1, 2, 10],
    labels=["Oldpeak=None", "Oldpeak=Mild", "Oldpeak=Moderate", "Oldpeak=High"]
)

# Make categorical labels explicit so the generated item names are easy to read.
arm_df["Sex"] = "Sex=" + arm_df["Sex"].astype(str)
arm_df["ChestPainType"] = "ChestPainType=" + arm_df["ChestPainType"].astype(str)
arm_df["FastingBS"] = arm_df["FastingBS"].map({0: "FastingBS=No", 1: "FastingBS=Yes"})
arm_df["RestingECG"] = "RestingECG=" + arm_df["RestingECG"].astype(str)
arm_df["ExerciseAngina"] = arm_df["ExerciseAngina"].map({"N": "ExerciseAngina=No", "Y": "ExerciseAngina=Yes"})
arm_df["ST_Slope"] = "ST_Slope=" + arm_df["ST_Slope"].astype(str)
arm_df["HeartDisease"] = arm_df["HeartDisease"].map({0: "HeartDisease=No", 1: "HeartDisease=Yes"})

arm_columns = [
    "Sex", "ChestPainType", "FastingBS", "RestingECG", "ExerciseAngina", "ST_Slope",
    "AgeGroup", "RestingBPGroup", "CholesterolGroup", "MaxHRGroup", "OldpeakGroup", "HeartDisease"
]

transactions = arm_df[arm_columns].astype(str)
transactions.head()

In [ ]:
# One-hot encode the transaction-style data.
# Each column becomes True or False depending on whether the item appears for that patient.
basket = pd.get_dummies(transactions)
basket = basket.astype(bool)

print(f"Transaction matrix shape: {basket.shape}")
basket.head()

In [ ]:
# Run Apriori and generate association rules.
# Support: how common the itemset is in the dataset.
# Confidence: how often the rule is correct when the antecedent appears.
# Lift: how much stronger the rule is than random chance. Lift greater than 1 is usually interesting.
if apriori is None or association_rules is None:
    print("mlxtend is not installed. Run this in a notebook cell if needed: !pip install mlxtend")
else:
    frequent_itemsets = apriori(basket, min_support=0.08, use_colnames=True)
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

    # Keep rules that point to heart disease presence.
    heart_rules = rules[rules["consequents"].apply(lambda x: "HeartDisease_HeartDisease=Yes" in x)].copy()
    heart_rules = heart_rules.sort_values(["lift", "confidence", "support"], ascending=False)

    display(heart_rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10))

### Association Rule Interpretation Guide

When reading the rules above:

- A higher **support** means the pattern appears in more patients.
- A higher **confidence** means the rule more often leads to `HeartDisease=Yes`.
- A **lift** greater than `1` means the combination is more associated with heart disease than random chance.

Rules should be interpreted as associations, not as proof that one factor causes another.

## 6. K-Means Clustering for Patient Archetypes

K-Means is an unsupervised learning method. It does not use the target label while creating groups. Instead, it groups patients based on similarity across features. We can then describe each cluster as a patient archetype.

In [ ]:
# Prepare features for clustering.
# We use the full feature set, but not the target column, because clustering is unsupervised.
X_cluster = X.copy()

cluster_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

X_cluster_encoded = cluster_preprocessor.fit_transform(X_cluster)

print(f"Encoded clustering feature matrix shape: {X_cluster_encoded.shape}")

In [ ]:
# Use the elbow method to compare different values of k.
# Inertia measures how tightly points fit within their clusters. Lower is better, but too many clusters can overcomplicate interpretation.
inertias = []
k_values = range(2, 9)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans.fit(X_cluster_encoded)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(k_values), inertias, marker="o")
plt.title("Elbow Method for Choosing k")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.xticks(list(k_values))
plt.show()

In [ ]:
# Choose k=3 for simple, interpretable patient archetypes.
# You may adjust this value after reviewing the elbow chart.
chosen_k = 3
kmeans_model = KMeans(n_clusters=chosen_k, random_state=RANDOM_STATE, n_init=10)
df["Cluster"] = kmeans_model.fit_predict(X_cluster_encoded)

df[["Age", "Sex", "ChestPainType", "MaxHR", "Oldpeak", "HeartDisease", "Cluster"]].head()

In [ ]:
# Describe each cluster using averages and heart disease rate.
cluster_profile_numeric = df.groupby("Cluster")[numeric_cols + ["HeartDisease"]].mean().round(2)
cluster_sizes = df["Cluster"].value_counts().sort_index().rename("PatientCount")

cluster_summary = cluster_profile_numeric.join(cluster_sizes)
cluster_summary = cluster_summary.rename(columns={"HeartDisease": "HeartDiseaseRate"})

display(cluster_summary)

In [ ]:
# View the most common categorical values inside each cluster.
for cluster_id in sorted(df["Cluster"].unique()):
    print(f"\nCluster {cluster_id} common categories:")
    cluster_data = df[df["Cluster"] == cluster_id]
    for col in categorical_cols:
        most_common = cluster_data[col].mode()[0]
        print(f"  {col}: {most_common}")

## 7. Decision Tree Classification

A Decision Tree is a supervised model that creates a tree of questions to classify patients. It is easy to interpret because we can visualize the decision path.

In [ ]:
# Build a Decision Tree pipeline.
# max_depth limits tree complexity and helps reduce overfitting.
decision_tree_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE))
])

decision_tree_pipeline.fit(X_train, y_train)

dt_predictions = decision_tree_pipeline.predict(X_test)
dt_probabilities = decision_tree_pipeline.predict_proba(X_test)[:, 1]

print("Decision Tree model trained successfully.")

In [ ]:
# Evaluate the Decision Tree using common classification metrics.
print("Decision Tree Classification Report")
print(classification_report(y_test, dt_predictions))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, dt_predictions), annot=True, fmt="d", cmap="Blues")
plt.title("Decision Tree Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Visualize the trained Decision Tree.
dt_model = decision_tree_pipeline.named_steps["model"]
dt_preprocessor = decision_tree_pipeline.named_steps["preprocessor"]
dt_feature_names = get_feature_names(dt_preprocessor)

plt.figure(figsize=(22, 10))
plot_tree(
    dt_model,
    feature_names=dt_feature_names,
    class_names=["No Heart Disease", "Heart Disease"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Decision Tree Visualization")
plt.show()

## 8. Random Forest Classification

A Random Forest is an ensemble model made of many decision trees. It often performs better than a single Decision Tree because it reduces overfitting by combining multiple trees.

In [ ]:
# Build a Random Forest pipeline.
random_forest_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        max_depth=None,
        class_weight="balanced"
    ))
])

random_forest_pipeline.fit(X_train, y_train)

rf_predictions = random_forest_pipeline.predict(X_test)
rf_probabilities = random_forest_pipeline.predict_proba(X_test)[:, 1]

print("Random Forest model trained successfully.")

In [ ]:
# Evaluate the Random Forest model.
print("Random Forest Classification Report")
print(classification_report(y_test, rf_predictions))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, rf_predictions), annot=True, fmt="d", cmap="Greens")
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Feature importance helps identify which variables were most useful to the Random Forest.
rf_model = random_forest_pipeline.named_steps["model"]
rf_preprocessor = random_forest_pipeline.named_steps["preprocessor"]
rf_feature_names = get_feature_names(rf_preprocessor)

feature_importance = pd.DataFrame({
    "feature": rf_feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance.head(10))

plt.figure(figsize=(9, 6))
sns.barplot(data=feature_importance.head(10), x="importance", y="feature")
plt.title("Top 10 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## 9. Model Evaluation and Comparison

We compare Decision Tree and Random Forest using accuracy, precision, recall, F1-score, and ROC-AUC. For heart disease screening, recall is especially important because missing a positive case can be clinically risky.

In [ ]:
# Build a comparison table for both classification models.
model_results = pd.DataFrame([
    {
        "Model": "Decision Tree",
        "Accuracy": accuracy_score(y_test, dt_predictions),
        "Precision": precision_score(y_test, dt_predictions),
        "Recall": recall_score(y_test, dt_predictions),
        "F1 Score": f1_score(y_test, dt_predictions),
        "ROC-AUC": roc_auc_score(y_test, dt_probabilities)
    },
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_test, rf_predictions),
        "Precision": precision_score(y_test, rf_predictions),
        "Recall": recall_score(y_test, rf_predictions),
        "F1 Score": f1_score(y_test, rf_predictions),
        "ROC-AUC": roc_auc_score(y_test, rf_probabilities)
    }
])

display(model_results.round(3))

In [ ]:
# Plot ROC curves for both models.
dt_fpr, dt_tpr, _ = roc_curve(y_test, dt_probabilities)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probabilities)

plt.figure(figsize=(7, 5))
plt.plot(dt_fpr, dt_tpr, label=f"Decision Tree AUC = {roc_auc_score(y_test, dt_probabilities):.3f}")
plt.plot(rf_fpr, rf_tpr, label=f"Random Forest AUC = {roc_auc_score(y_test, rf_probabilities):.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Guess")
plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

In [ ]:
# Automatically identify the stronger model by F1 score.
# F1 balances precision and recall, making it useful when both false positives and false negatives matter.
best_model_row = model_results.sort_values("F1 Score", ascending=False).iloc[0]
print(f"Best model by F1 Score: {best_model_row['Model']}")
print(best_model_row.round(3))

## 10. Final Clinical Insights and Recommendation

This section summarizes the most important findings from the data mining process. These statements should be updated after running the notebook and reviewing the actual outputs.

In [ ]:
# Create a small summary of data-driven signals that can support the final written interpretation.
heart_disease_rate = df["HeartDisease"].mean() * 100
top_rf_features = feature_importance.head(5)["feature"].tolist()

print(f"Overall heart disease rate in the dataset: {heart_disease_rate:.2f}%")
print("Top Random Forest features:")
for feature in top_rf_features:
    print(f"- {feature}")

print("\nCluster heart disease rates:")
display(cluster_summary[["PatientCount", "HeartDiseaseRate"]])

### Final Insights

After running the analysis, the project should support several clinical-style insights:

1. **Patient risk is not explained by one variable alone.** Heart disease patterns are better understood by combining symptoms, exercise response, heart rate, age, and ECG-related information.
2. **Association rules can highlight common risk profiles.** Rules with high lift and confidence may reveal combinations such as specific chest pain types, exercise-induced angina, ST slope patterns, or older age groups that appear more often with heart disease.
3. **K-Means clustering helps describe patient archetypes.** The clusters can separate patients into groups with different average ages, heart rates, Oldpeak values, and heart disease rates.
4. **Random Forest often provides stronger predictive performance.** Compared with a single Decision Tree, Random Forest usually gives more stable results because it combines many trees.
5. **Decision Trees remain useful for explanation.** Even if the Random Forest performs better, the Decision Tree visualization is easier to explain to beginners and non-technical readers.

### Recommendation

For this educational project, the recommended final model is the model with the better F1-score and ROC-AUC after running the evaluation cells. If Random Forest performs best, it should be used for prediction, while the Decision Tree and association rules should be used to explain important patterns in a more interpretable way.

In a real clinical setting, a model like this should only support decision-making. It should be validated with larger and more diverse patient data, reviewed by healthcare professionals, and combined with proper clinical assessment.